In [1]:
!rocm-smi

======================= ROCm System Management Interface =======================
================================= Concise Info =================================
GPU  Temp (DieEdge)  AvgPwr  SCLK    MCLK    Fan  Perf  PwrCap  VRAM%  GPU%
0    45.0c           65.0W   800Mhz  1600Mhz 0%   auto  750.0W   12%   0% 
============================= End of ROCm SMI Log ==============================


In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"ROCm available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.4.0+rocm6.3
ROCm available: True
Device: AMD Instinct MI300X


# Notebook 04: XGBoost + SMOTE-ENN + Optuna

- Per-head XGBoost classifiers (surface, transparency, color_family)
- 339+ features: UMF oxides, ingredient TF-IDF, chemical composition, domain ratios, atmosphere, cone
- SMOTE-ENN to rescue rare classes (Purple, Orange, Dry Matte, Stony Matte)
- Optuna Bayesian hyperparameter optimization (100 trials per head)
- SHAP feature importance analysis
- Target: 58% avg accuracy / 44% macro F1 (nb01 baseline: 48.3%/31.6%)


In [27]:
import sys, subprocess, warnings, os, re, json, time
import numpy as np
import pandas as pd

try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', 'optuna-integration', 'imbalanced-learn', 'xgboost', 'scikit-learn', 'pandas'])
    import optuna

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
print(f'Optuna {optuna.__version__}  XGBoost {XGBClassifier.__module__}')

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.7/615.7 kB 126.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [optuna-integration]mbalanced-learn]



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/venv/bin/python3.10 -m pip install --upgrade pip


Optuna 4.9.0  XGBoost xgboost.sklearn


## 1. Load raw data


In [28]:
DATA_DIR = 'data/property_prediction'
assert os.path.exists(DATA_DIR), f'Upload data to {DATA_DIR}'

def jload(path):
    with open(os.path.join(DATA_DIR, path)) as f:
        return json.load(f)

train_recipes = jload('train/recipes.json')
train_targets = jload('train/targets.json')
test_recipes  = jload('test/recipes.json')
test_targets  = jload('test/targets.json')
print(f'Train: {len(train_recipes)} recipes / {len(train_targets)} targets')
print(f'Test:  {len(test_recipes)} recipes / {len(test_targets)} targets')

Train: 16781 recipes / 16781 targets
Test:  4903 recipes / 4903 targets


## 2. Build DataFrame with all feature types


In [29]:
def build_df(recipes, targets):
    tmap = {t['id']: t for t in targets}
    rows = []
    for r in recipes:
        t = tmap.get(r['id'])
        if t is None:
            continue
        row = {'id': r['id']}
        # UMF oxides
        umf = r.get('umf') or {}
        for k, v in umf.items():
            row['umf_' + k] = v
        # Chemical composition (46 oxides - used as features)
        cc = r.get('chemical_composition') or {}
        for k, v in cc.items():
            if isinstance(v, (int, float)):
                row['chem_' + k] = v
        # Cone
        row['cone_min'] = r.get('cone_min')
        row['cone_max'] = r.get('cone_max')
        # Atmosphere
        row['atmosphere'] = r.get('atmosphere', '')
        # Misc
        row['has_chem'] = float(r.get('has_chemical_data', False))
        row['ing_count'] = r.get('ingredient_count', 0)
        row['batch_total'] = cc.get('Amt.', 0) or 0
        row['loi'] = cc.get('LOI', 0) or 0
        # Ingredients text
        ings = r.get('ingredients', [])
        row['ing_text'] = ' '.join(ing.get('material', '') for ing in ings)
        # Targets
        for h in ('surface', 'transparency', 'color_family'):
            v = t.get(h)
            if v is not None and isinstance(v, str):
                row[h] = v
        # Color RGB
        rgb = t.get('color_rgb')
        if rgb:
            row['color_r'] = rgb.get('r') / 255.0
            row['color_g'] = rgb.get('g') / 255.0
            row['color_b'] = rgb.get('b') / 255.0
        rows.append(row)
    return pd.DataFrame(rows)

df_train = build_df(train_recipes, train_targets)
df_test  = build_df(test_recipes,  test_targets)

umf_cols = [c for c in df_train.columns if c.startswith('umf_')]
df_train = df_train[df_train[umf_cols].notna().any(axis=1)].reset_index(drop=True)
df_test  = df_test[df_test[[c for c in df_test.columns if c.startswith('umf_')]].notna().any(axis=1)].reset_index(drop=True)
print(f'Train: {df_train.shape}  Test: {df_test.shape}')

Train: (15668, 79)  Test: (4849, 69)


### 2a. Ingredient TF-IDF


In [30]:
all_ing = pd.concat([df_train['ing_text'], df_test['ing_text']])
vec = CountVectorizer(token_pattern=r"(?u)\b\w+['\-]?\w+\b", min_df=20, max_features=300)
vec.fit(all_ing)
for df in (df_train, df_test):
    mat = vec.transform(df['ing_text']).toarray()
    for i, c in enumerate(vec.get_feature_names_out()):
        df['ing_' + c] = mat[:, i]
print(f'Ingredient features: {len(vec.get_feature_names_out())}')

Ingredient features: 300


### 2b. Domain chemistry ratios


In [31]:
def add_domain_ratios(df):
    def g(k):
        c = 'umf_' + k
        return df[c] if c in df.columns else 0.0
    si, al = g('SiO2').clip(lower=0), g('Al2O3').clip(lower=1e-6)
    df['r_si_al'] = si / al
    flux_oxides = ['Na2O','K2O','CaO','MgO','ZnO','Li2O','PbO','BaO','SrO']
    fcols = [c for c in df.columns if c.startswith('umf_') and c[4:] in flux_oxides]
    df['r_total_flux'] = df[fcols].sum(axis=1) if fcols else 0.0
    alkali  = ['Na2O','K2O','Li2O']
    ae      = ['CaO','MgO','BaO','SrO']
    acols  = [c for c in df.columns if c.startswith('umf_') and c[4:] in alkali]
    aecols = [c for c in df.columns if c.startswith('umf_') and c[4:] in ae]
    df['r_alkali']       = df[acols].sum(axis=1) if acols else 0.0
    df['r_alkali_earth'] = df[aecols].sum(axis=1) if aecols else 0.0
    at = df['r_alkali'] + df['r_alkali_earth'].clip(lower=1e-6)
    df['r_alkali_ratio'] = df['r_alkali'] / at
    na = g('Na2O').clip(lower=1e-6)
    df['r_na_k'] = g('K2O') / na
    colorants = ['Fe2O3','CuO','CoO','Cr2O3','MnO2','TiO2','MnO','NiO','V2O5']
    ccols = [c for c in df.columns if c.startswith('umf_') and c[4:] in colorants]
    df['r_colorant_load'] = df[ccols].sum(axis=1) if ccols else 0.0
    tf = df['r_total_flux'].clip(lower=1e-6)
    df['r_si_flux'] = si / tf
    df['r_b2o3_present'] = (g('B2O3') > 0).astype(float)
    df['r_pbo_present']  = (g('PbO') > 0).astype(float)
    df['r_fe_total']     = g('Fe2O3')
    df.fillna(0.0, inplace=True)

add_domain_ratios(df_train)
add_domain_ratios(df_test)

### 2c. Atmosphere one-hot + cone clipping


In [32]:
for df in (df_train, df_test):
    atmos_dummies = pd.get_dummies(df['atmosphere'], prefix='atmos')
    for c in atmos_dummies.columns:
        df[c] = atmos_dummies[c].astype(float)
    df['cone_max'] = df['cone_max'].clip(upper=22)
    df['cone_min'] = df['cone_min'].clip(upper=22)

### 2d. Assemble feature column list


In [33]:
FEATURE_COLS = (
    [c for c in df_train.columns if c.startswith('umf_')] +
    [c for c in df_train.columns if c.startswith('chem_')] +
    [c for c in df_train.columns if c.startswith('ing_') and c != 'ing_text'] +
    [c for c in df_train.columns if c.startswith('r_')] +
    [c for c in df_train.columns if c.startswith('atmos_')] +
    ['cone_min', 'cone_max', 'has_chem', 'ing_count', 'batch_total', 'loi']
)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_train.columns and df_train[c].dtype in (np.float64, np.float32, np.int64, np.int32)]
print(f'Total features: {len(FEATURE_COLS)}')
print(f'  UMF: {sum(1 for c in FEATURE_COLS if c.startswith("umf_"))}')
print(f'  Chem: {sum(1 for c in FEATURE_COLS if c.startswith("chem_"))}')
print(f'  Ing: {sum(1 for c in FEATURE_COLS if c.startswith("ing_"))}')
print(f'  Dom ratios: {sum(1 for c in FEATURE_COLS if c.startswith("r_"))}')
print(f'  Atmos: {sum(1 for c in FEATURE_COLS if c.startswith("atmos_"))}')
print(f'  Other: {sum(1 for c in FEATURE_COLS if not c.startswith(("umf_","chem_","ing_","r_","atmos_")))}')

Total features: 386
  UMF: 18
  Chem: 46
  Ing: 302
  Dom ratios: 11
  Atmos: 4
  Other: 5


### 2e. Fill missing + scale


In [34]:
scaler = StandardScaler()
for col in FEATURE_COLS:
    if col not in df_train.columns: df_train[col] = 0.0
    if col not in df_test.columns:  df_test[col]  = 0.0
    df_train[col] = df_train[col].fillna(0.0)
    df_test[col]  = df_test[col].fillna(0.0)
df_train[FEATURE_COLS] = scaler.fit_transform(df_train[FEATURE_COLS])
df_test[FEATURE_COLS]  = scaler.transform(df_test[FEATURE_COLS])
print('Scaled')

Scaled


## 3. Per-head datasets + SMOTE-ENN


In [35]:
HEADS = ['surface', 'transparency', 'color_family']
HEAD_DATA = {}

for h in HEADS:
    mask_tr = df_train[h].notna() & df_train[h].apply(lambda x: isinstance(x, str))
    mask_te = df_test[h].notna() & df_test[h].apply(lambda x: isinstance(x, str))

    le = LabelEncoder()
    y_tr = le.fit_transform(df_train.loc[mask_tr, h])
    y_te = le.transform(df_test.loc[mask_te, h])
    X_tr = df_train.loc[mask_tr, FEATURE_COLS].values.astype(np.float32)
    X_te = df_test.loc[mask_te, FEATURE_COLS].values.astype(np.float32)

    # SMOTE-ENN for surface and color_family (heavy imbalance)
    apply_smote = h != 'transparency'
    if apply_smote:
        smote = SMOTEENN(random_state=42, n_jobs=-1)
        X_res, y_res = smote.fit_resample(X_tr, y_tr)
        print(f'{h:15s}  original: {X_tr.shape[0]:5d}  after SMOTEENN: {X_res.shape[0]:5d}  classes: {len(np.unique(y_res))}')
        counts = np.bincount(y_res)
        print(f'  class distribution: min={counts.min():4d}  max={counts.max():4d}  mean={counts.mean():.0f}')
    else:
        X_res, y_res = X_tr, y_tr
        print(f'{h:15s}  original: {X_tr.shape[0]:5d}  (no SMOTE)  classes: {len(np.unique(y_res))}')

    # Class weights (inverse frequency, computed BEFORE SMOTE to avoid biasing)
    counts_orig = np.bincount(y_tr)
    sw = len(y_tr) / (len(counts_orig) * counts_orig.astype(float))
    sw = sw / sw.sum() * len(counts_orig)
    sample_weight_dict = dict(enumerate(sw))
    sw_res = np.array([sample_weight_dict[i] for i in y_res])

    HEAD_DATA[h] = {
        'le': le,
        'X_tr': X_tr, 'y_tr': y_tr,
        'X_te': X_te, 'y_te': y_te,
        'X_res': X_res, 'y_res': y_res,
        'sw_res': sw_res,
        'n': len(le.classes_),
        'classes': le.classes_,
        'tr_idx': np.where(mask_tr.values)[0],
        'te_idx': np.where(mask_te.values)[0],
    }
    del le, X_tr, y_tr, X_te, y_te, X_res, y_res, sw_res, mask_tr, mask_te

surface          original:  9367  after SMOTEENN: 35439  classes: 9
  class distribution: min=1683  max=4525  mean=3938
transparency     original:  9013  (no SMOTE)  classes: 4
color_family     original: 15668  after SMOTEENN: 31941  classes: 9
  class distribution: min= 719  max=5260  mean=3549


## 4. Optuna hyperparameter optimization


In [37]:
N_TRIALS = 100
CV_FOLDS = 3

best_params = {}
optuna_studies = {}

for h in HEADS:
    d = HEAD_DATA[h]
    print(f'\n{"="*60}\n  Optuna: {h} ({d["n"]} classes, {d["X_res"].shape[0]} samples)  \n{"="*60}')

    def objective(trial, head=h, data=d):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 800, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'gamma': trial.suggest_float('gamma', 0.0, 2.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 10.0, log=True),
        }
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
        scores = []
        for tr_i, va_i in skf.split(data['X_res'], data['y_res']):
            X_tr, X_va = data['X_res'][tr_i], data['X_res'][va_i]
            y_tr, y_va = data['y_res'][tr_i], data['y_res'][va_i]
            sw_tr = data['sw_res'][tr_i]
            m = XGBClassifier(
                **params,
                objective='multi:softprob',
                eval_metric='mlogloss',
                num_class=data['n'],
                random_state=42,
                n_jobs=-1,
                verbosity=0,
            )
            m.fit(X_tr, y_tr, sample_weight=sw_tr, verbose=False)
            scores.append(f1_score(y_va, m.predict(X_va), average='macro'))
        return float(np.mean(scores))

    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5),
        study_name=h,
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_params[h] = study.best_params
    optuna_studies[h] = study
    print(f'  Best score: {study.best_value:.4f}')
    print(f'  Best params: {study.best_params}')

[I 2026-07-09 13:46:26,840] A new study created in memory with name: surface



  Optuna: surface (9 classes, 35439 samples)  


Best trial: 0. Best value: 0.859634:   1%|          | 1/100 [01:29<2:27:00, 89.10s/it]

[I 2026-07-09 13:47:55,940] Trial 0 finished with value: 0.8596337962573285 and parameters: {'n_estimators': 350, 'max_depth': 15, 'learning_rate': 0.1001303991139125, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.40921304830970556, 'min_child_weight': 4, 'gamma': 0.11616722433639892, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.027182927734624}. Best is trial 0 with value: 0.8596337962573285.


Best trial: 0. Best value: 0.859634:   2%|▏         | 2/100 [02:12<1:41:17, 62.01s/it]

...

[I 2026-07-09 19:50:46,719] Trial 97 finished with value: 0.8110464617989966 and parameters: {'n_estimators': 700, 'max_depth': 7, 'learning_rate': 0.26010616494808486, 'subsample': 0.7760993033740831, 'colsample_bytree': 0.49200822712711434, 'min_child_weight': 15, 'gamma': 0.048151392446166646, 'reg_alpha': 0.1341764551461378, 'reg_lambda': 3.5660079341810835}. Best is trial 94 with value: 0.8254160933182101.


Best trial: 94. Best value: 0.825416:  99%|█████████▉| 99/100 [2:39:19<01:33, 93.43s/it]

[I 2026-07-09 19:52:25,751] Trial 98 finished with value: 0.8114143669704431 and parameters: {'n_estimators': 750, 'max_depth': 6, 'learning_rate': 0.25992166400728334, 'subsample': 0.8337926516876814, 'colsample_bytree': 0.38942520711445977, 'min_child_weight': 16, 'gamma': 0.05145975031373682, 'reg_alpha': 0.0003115007862872247, 'reg_lambda': 3.731528060087412}. Best is trial 94 with value: 0.8254160933182101.


Best trial: 94. Best value: 0.825416: 100%|██████████| 100/100 [2:40:19<00:00, 96.20s/it]

[I 2026-07-09 19:53:25,950] Trial 99 finished with value: 0.7251114628654266 and parameters: {'n_estimators': 800, 'max_depth': 5, 'learning_rate': 0.22752297954242942, 'subsample': 0.8351385936635153, 'colsample_bytree': 0.3141587358505719, 'min_child_weight': 18, 'gamma': 0.2463516497639625, 'reg_alpha': 0.0013478638026281596, 'reg_lambda': 3.5766873544952036}. Best is trial 94 with value: 0.8254160933182101.
  Best score: 0.8254
  Best params: {'n_estimators': 750, 'max_depth': 7, 'learning_rate': 0.26892128938957693, 'subsample': 0.7755668932036138, 'colsample_bytree': 0.48794028588770516, 'min_child_weight': 11, 'gamma': 0.062257983617797066, 'reg_alpha': 0.014742238919825998, 'reg_lambda': 5.555049323584537}


## 5. Train final models with best params


In [38]:
final_models = {}

for h in HEADS:
    d = HEAD_DATA[h]
    bp = best_params[h]
    evalset = [(d['X_res'], d['y_res'])]

    model = XGBClassifier(
        **bp,
        objective='multi:softprob',
        eval_metric='mlogloss',
        num_class=d['n'],
        random_state=42,
        n_jobs=-1,
        verbosity=0,
        early_stopping_rounds=30,
    )
    model.fit(
        d['X_res'], d['y_res'],
        sample_weight=d['sw_res'],
        eval_set=evalset,
        verbose=False,
    )
    final_models[h] = model
    print(f'{h:15s}  trained ({len(model.get_booster().get_dump())} trees)')

surface          trained (7200 trees)
transparency     trained (1600 trees)
color_family     trained (6750 trees)


## 6. Feature importance analysis


In [39]:
print(f'{"Head":<15s} {"Top-5 features (gain)":<60s}')
print('-' * 75)
for h in HEADS:
    m = final_models[h]
    imp = pd.DataFrame({'feature': FEATURE_COLS, 'gain': m.feature_importances_})
    imp = imp.sort_values('gain', ascending=False).head(5)
    top5 = ', '.join(f'{r.feature} ({r.gain:.3f})' for _, r in imp.iterrows())
    print(f'{h:<15s}  {top5}')

    # Top-30 per head (collect for plotting)
    imp_all = pd.DataFrame({'feature': FEATURE_COLS, 'gain': m.feature_importances_})
    imp_all.to_csv(f'/tmp/featimp_{h}.csv', index=False)

Head            Top-5 features (gain)                                       
---------------------------------------------------------------------------
surface          ing_rutil (0.055), ing_puraflo (0.023), ing_3249 (0.021), ing_ultrafine (0.016), ing_оксид (0.015)
transparency     ing_st20 (0.013), r_colorant_load (0.013), ing_ron (0.010), ing_super (0.010), ing_tin (0.009)
color_family     ing_slip (0.063), ing_кварцевая (0.040), ing_14 (0.039), ing_green (0.035), ing_silverbond (0.027)


## 7. Test evaluation


In [40]:
test_results = {}

for h in HEADS:
    d = HEAD_DATA[h]
    y_pred = final_models[h].predict(d['X_te'])
    test_results[h] = {
        'accuracy': accuracy_score(d['y_te'], y_pred),
        'macro_f1': f1_score(d['y_te'], y_pred, average='macro'),
        'weighted_f1': f1_score(d['y_te'], y_pred, average='weighted'),
        'preds': y_pred,
    }

print(f'{"Head":<15s} {"Accuracy":>10s} {"Macro F1":>10s} {"Wtd F1":>10s}')
print('-' * 50)
for h in HEADS:
    r = test_results[h]
    print(f'{h:<15s}  {r["accuracy"]:.4f}  {r["macro_f1"]:.4f}  {r["weighted_f1"]:.4f}')
avg_acc = np.mean([test_results[h]['accuracy'] for h in HEADS])
avg_f1  = np.mean([test_results[h]['macro_f1'] for h in HEADS])
print(f'{"Average":<15s}  {avg_acc:.4f}  {avg_f1:.4f}')

Head              Accuracy   Macro F1     Wtd F1
--------------------------------------------------
surface          0.3849  0.2776  0.4099
transparency     0.5973  0.5432  0.5847
color_family     0.3308  0.3089  0.3130
Average          0.4377  0.3766


## 8. Per-class breakdown


In [41]:
for h in HEADS:
    d = HEAD_DATA[h]
    print(f'\n=== {h} ===')
    print(classification_report(
        d['y_te'], test_results[h]['preds'],
        target_names=d['classes'], digits=4,
    ))
    cm = confusion_matrix(d['y_te'], test_results[h]['preds'])
    print(f'Confusion matrix ({h}):')
    print(np.array2string(cm, max_line_width=120))


=== surface ===
              precision    recall  f1-score   support

   Dry Matte     0.0923    0.1818    0.1224        33
      Glossy     0.8155    0.4102    0.5458      1692
       Matte     0.3864    0.5163    0.4420       461
       Satin     0.2226    0.2012    0.2114       323
 Satin-matte     0.2475    0.2641    0.2555       284
 Semi-glossy     0.2458    0.4917    0.3278       539
  Semi-matte     0.1837    0.2332    0.2055       223
Smooth Matte     0.1720    0.2213    0.1935       122
 Stony Matte     0.1566    0.2549    0.1940        51

    accuracy                         0.3849      3728
   macro avg     0.2803    0.3083    0.2776      3728
weighted avg     0.5112    0.3849    0.4099      3728

Confusion matrix (surface):
[[  6   1  16   1   2   0   1   1   5]
 [  4 694 107  89  93 588  83  29   5]
 [ 30   7 238  29  38  21  32  40  26]
 [  5  41  42  65  21  97  30  13   9]
 [  4   8  63  33  75  49  26  18   8]
 [  5  86  36  50  31 265  46  16   4]
 [  5   9  64  1

## 9. Comparison vs baseline (notebook 01)


In [42]:
# Baseline from notebook 01 (22 features, no SMOTE, no Optuna, fixed depth=6)
BASELINE = {
    'surface':      {'accuracy': 0.5491, 'macro_f1': 0.2652},
    'transparency': {'accuracy': 0.5608, 'macro_f1': 0.4745},
    'color_family': {'accuracy': 0.4533, 'macro_f1': 0.3126},
}

print(f'{"Head":<15s} {"Old Acc":>8s} {"New Acc":>8s} {"Δ Acc":>8s}  {"Old F1":>8s} {"New F1":>8s} {"Δ F1":>8s}')
print('-' * 70)
acc_gains = []
f1_gains = []
for h in HEADS:
    old = BASELINE[h]
    new = test_results[h]
    da = new['accuracy'] - old['accuracy']
    df1 = new['macro_f1'] - old['macro_f1']
    acc_gains.append(da)
    f1_gains.append(df1)
    print(f'{h:<15s}  {old["accuracy"]:.4f}  {new["accuracy"]:.4f}  {da:+.4f}  '
          f'{old["macro_f1"]:.4f}  {new["macro_f1"]:.4f}  {df1:+.4f}')

old_avg = np.mean([BASELINE[h]['accuracy'] for h in HEADS])
new_avg = np.mean([test_results[h]['accuracy'] for h in HEADS])
old_f1_avg = np.mean([BASELINE[h]['macro_f1'] for h in HEADS])
new_f1_avg = np.mean([test_results[h]['macro_f1'] for h in HEADS])
print(f'{"Average":<15s}  {old_avg:.4f}  {new_avg:.4f}  {new_avg-old_avg:+.4f}  '
      f'{old_f1_avg:.4f}  {new_f1_avg:.4f}  {new_f1_avg-old_f1_avg:+.4f}')

Head             Old Acc  New Acc    Δ Acc    Old F1   New F1     Δ F1
----------------------------------------------------------------------
surface          0.5491  0.3849  -0.1642  0.2652  0.2776  +0.0124
transparency     0.5608  0.5973  +0.0365  0.4745  0.5432  +0.0687
color_family     0.4533  0.3308  -0.1225  0.3126  0.3089  -0.0037
Average          0.5211  0.4377  -0.0834  0.3508  0.3766  +0.0258


## 10. Optuna study summary


In [43]:
for h in HEADS:
    study = optuna_studies[h]
    print(f'\n{h} Optuna results:')
    print(f'  Best score: {study.best_value:.4f}')
    print(f'  Best params:')
    for k, v in study.best_params.items():
        print(f'    {k}: {v}')
    print(f'  Completed trials: {len(study.trials)}')
    print(f'  Pruned trials: {sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)}')


surface Optuna results:
  Best score: 0.9243
  Best params:
    n_estimators: 800
    max_depth: 13
    learning_rate: 0.06668718016236205
    subsample: 0.836842544695184
    colsample_bytree: 0.46862732752246117
    min_child_weight: 2
    gamma: 0.04569900611216655
    reg_alpha: 1.4535931274949918
    reg_lambda: 2.3850749078263744
  Completed trials: 100
  Pruned trials: 0

transparency Optuna results:
  Best score: 0.6169
  Best params:
    n_estimators: 400
    max_depth: 14
    learning_rate: 0.07676127973322823
    subsample: 0.9343193395993378
    colsample_bytree: 0.5678997032747912
    min_child_weight: 1
    gamma: 0.08384575443178269
    reg_alpha: 1.7912121831529055
    reg_lambda: 9.152637082798433
  Completed trials: 100
  Pruned trials: 0

color_family Optuna results:
  Best score: 0.8254
  Best params:
    n_estimators: 750
    max_depth: 7
    learning_rate: 0.26892128938957693
    subsample: 0.7755668932036138
    colsample_bytree: 0.48794028588770516
    min_chil